## Agents

In [1]:
import os
from dotenv import load_dotenv
from langchain_classic.agents import create_react_agent, AgentExecutor
from langchain_classic.tools import DuckDuckGoSearchRun
from langchain_classic import hub

from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_groq import ChatGroq

W0526 01:58:05.760000 7979 site-packages/torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W0526 01:58:05.784000 7979 site-packages/torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


In [2]:
load_dotenv()

True

In [3]:
## 1. Initializing LLM and Search Tool
llm_groq = ChatGroq(model='meta-llama/llama-4-scout-17b-16e-instruct',streaming=True)
llm_ollama = ChatOllama(model='qwen3-vl:235b-instruct-cloud',streaming=True)

## Search Tool
search_tool = DuckDuckGoSearchRun()

In [4]:
## 2. Pulling React Prompt from LangChain Hub

prompt = hub.pull("hwchase17/react")
prompt

/tmp/ipykernel_7979/1646415371.py:3: LangChainDeprecationWarning: langchain_classic.hub.pull is deprecated. Use the LangSmith SDK instead.
  prompt = hub.pull("hwchase17/react")


PromptTemplate(input_variables=['agent_scratchpad', 'input', 'tool_names', 'tools'], input_types={}, partial_variables={}, metadata={'lc_hub_owner': 'hwchase17', 'lc_hub_repo': 'react', 'lc_hub_commit_hash': 'd15fe3c426f1c4b3f37c9198853e4a86e20c425ca7f4752ec0c9b0e97ca7ea4d'}, template='Answer the following questions as best you can. You have access to the following tools:\n\n{tools}\n\nUse the following format:\n\nQuestion: the input question you must answer\nThought: you should always think about what to do\nAction: the action to take, should be one of [{tool_names}]\nAction Input: the input to the action\nObservation: the result of the action\n... (this Thought/Action/Action Input/Observation can repeat N times)\nThought: I now know the final answer\nFinal Answer: the final answer to the original input question\n\nBegin!\n\nQuestion: {input}\nThought:{agent_scratchpad}')

In [5]:
## 3. Create React Agent using the pulled prompt
agent = create_react_agent(
    llm=llm_groq,
    tools=[search_tool],
    prompt=prompt
)

In [6]:
## 4. Wrapping it with AgentExecutor

agent_executor = AgentExecutor(
    agent=agent,
    tools=[search_tool],
    handle_parsing_errors=True,
    verbose=True
)

In [7]:
## 5. Invoke Agent

response = agent_executor.invoke(input={"input":"3 ways to reach Goa from Delhi."})
response



> Entering new AgentExecutor chain...
To answer this question, I need to find information on traveling from Delhi to Goa. 

Action: duckduckgo_search
Action Input: Delhi to Goa travel options12 hours ago -Direct flights from New Delhi to Goa are offered by several airlines, includingAir-India Express, Akasa Air, IndiGo, Air India. These flights usually take around 1h 50m. Connecting flights will typically add another 1-2 hours onto your travel ... 11 September 2025 -Book your Delhi to Goa flights at the lowest airfare with Akbar Travels. Enjoy exclusive discounts on domestic flight bookings — save up to ₹20,000 instantly with promo code “ATFLY”. Looking for the return journey? 24 September 2025 -This guide takes a closer look at the best route from Delhi to Goa by car, the Delhi to Goa road trip cost, suggested itineraries, weather, and even accommodation options like booking a villa in Goa once you arrive. ... The Delhi to Goa by road distance is approximately 1,850 to 1,950 kilomet

{'input': '3 ways to reach Goa from Delhi.',
 'output': 'Here are three ways to reach Goa from Delhi:\n\n1. **By Flight**: Flights from Delhi to Goa are available with several airlines, including Air India, IndiGo, and Akasa Air. The flight duration is approximately 2 hours and 20 minutes to 2 hours and 45 minutes.\n\n2. **By Train**: Trains from Delhi to Goa are available, with major stations like New Delhi (NDLS), Hazrat Nizamuddin, and Old Delhi station serving as railway hubs. \n\n3. **By Bus or Road**: Taking a bus or driving from Delhi to Goa is another option. The distance is approximately 1,850 to 1,950 kilometers, and it generally takes around 32 to 36 hours of continuous driving. Many travelers break the journey into three or four days to see attractions along the way. The cost of driving is approximately ₹18,088 for fuel and tolls.'}

In [8]:
response = agent_executor.invoke({"input":"Give Short summary about the IPL match held Gujarat Titans and Chennai Super Kings."})
response



> Entering new AgentExecutor chain...
To provide a short summary about the IPL match held between Gujarat Titans and Chennai Super Kings, I need to find information about their recent match. The best course of action would be to search for the latest news or match summary.

Action: duckduckgo_search
Action Input: Gujarat Titans vs Chennai Super Kings IPL match summary4 days ago ·FollowIPL| CSK 140vsGT 229/4 (Mukesh Choudhary 2(2) ) |GujaratTitansvsChennaiSuperKings, 66thMatch, Indian Premier League 2026 live with live scores, ball-by-ball commentary, highlights ... vs live score: get full scorecard of live cricket score vsmatchscorecard along with ball by ball commentary, player and team stats at Cricket News 4 days ago ·Catch the LIVE Cricket Score of the Match 66 betweenGujaratTitansandChennaiSuperKings. Get ball-by-ball updates, full scorecard, expert commentary & analysis on Firstpost. 4 days ago ·GujaratTitansvsChennaiSuperKingsHighlights,IPL2026: In anIPL2026 encounter in Ahmed

{'input': 'Give Short summary about the IPL match held Gujarat Titans and Chennai Super Kings.',
 'output': 'In the IPL 2026 encounter, Gujarat Titans defeated Chennai Super Kings by 89 runs. The match took place at the Narendra Modi Stadium in Ahmedabad. Gujarat Titans posted a score of 229/4, thanks to half-centuries from Shubman Gill and Sai Sudharsan, who combined for 125 runs for the first wicket. Chennai Super Kings could only manage to score 140. This victory was significant for Gujarat Titans, while Chennai Super Kings, led by Ruturaj Gaikwad, faced a tough challenge, sitting sixth on the table and desperately hunting for a win.'}

In [9]:
response = agent_executor.invoke({"input":"Between what dates I can ask questions?"})
response



> Entering new AgentExecutor chain...
Thought: I need to find out the specific dates during which I can ask questions, but I don't have any information about it. I should search for the knowledge cutoff and availability of the model.

Action: duckduckgo_search
Action Input: "knowledge cutoff date"Newknowledgecutoffdate...Knowledgecutoff: 2022-01 Currentdate: 2023-09-19" ... ModelKnowledgeCutoffDate(gpt-4-0125-preview) the model seems to be thinking that its’knowledgecutoffdateis still April 2023, contrary to what the documentation said about December 2023. For 2 days it was able to find it and knew it existed, suddenly yesterday though it seemed to forget, and theknowledgecutoffdatewas displayed. I am currently working with the text-embedding-3-large model and would like clarification on itsknowledgecutoffdate. Newknowledgecutoffdate... I was thinking thisKnowledgeCutoffis hallucination because ... ModelKnowledgeCutoffDate(gpt-4-0125-preview)The search results are not providing clear

{'input': 'Between what dates I can ask questions?',
 'output': 'You can ask me questions anytime, as I am a chatbot and do not have specific availability dates. However, my knowledge cutoff is December 2023, and I may not have information on events or topics that have occurred after that date.'}

In [10]:
response = agent_executor.invoke({"input":"Please tell me today's date"})
response



> Entering new AgentExecutor chain...
Thought: To find today's date, I can use the duckduckgo_search tool to search for the current date.

Action: duckduckgo_search
Action Input: "what is today's date"The exacttoday'sday anddateis... Throw the first stone who has never been lost in time after a big hangover and forgetwhatisthedatetoday. Thisisthe spelling of the fulldatetodaythatisclear for the majority of the countries worldwide. ... correct when writingtoday’sdate... Whatistoday’sdate? ...Whatwas yesterday ’sdate? /Whatistomorrow ’sdate? Next leap year? /Datetwo weeks from now? Knowingwhatthedateistodayprovides a clear starting point for tracking your progress for personal and professional goals, making it an ... TodaysDate🗓️WhatDayIsItToday? 🕐 CurrentDateand Time 📋DateFormats ... Knowingtodaysdateisessential for scheduling ...Thought: The search results don't provide a clear and direct answer to today's date. I'll try a more specific search query.

Action: duckduckgo_search
Action

{'input': "Please tell me today's date",
 'output': "I'm unable to provide today's date as the search results did not provide a clear and direct answer. However, I can suggest checking a reliable online source, such as a calendar or date website, for the most up-to-date and accurate information."}

In [13]:
response = agent_executor.invoke({"input":"Give Short summary about the recent IPL2026 match held between Gujarat Titans and Chennai Super Kings."})
response['output']



> Entering new AgentExecutor chain...
Thought: To provide a summary of the recent IPL 2026 match between Gujarat Titans and Chennai Super Kings, I first need to find information about this specific match. Since the match is in the future (IPL 2026), it's likely that the information isn't available yet. However, I can try searching for any recent matches or news related to these teams.

Action: duckduckgo_search
Action Input: Gujarat Titans vs Chennai Super Kings IPL 2026 recent match summary4 days ago ·Follow our live cricket update for in-depthmatchcoverage and exciting highlights fromGujaratTitansvsChennaiSuperKings66th Match at Ahmedabad on ESPNcricinfo. 4 days ago ·FollowIPL|GujaratTitansvsChennaiSuperKings, 66th Match, Indian PremierLeague2026live with live scores, ball-by-ball commentary, highlights, videos, news, and more on... 4 days ago ·GT vs CSK Highlights,IPL2026: Opener Sai Sudharsan and captain Shubman Gill struck fluent fifties to powerGujaratTitansto a commanding 89-r

"Gujarat Titans recently defeated Chennai Super Kings by 89 runs in their IPL 2026 match held at Ahmedabad. The match, which was part of the Indian Premier League 2026, saw Gujarat Titans post a total of 229 for four, thanks to fluent fifties from opener Sai Sudharsan and captain Shubman Gill. Their partnership of 125 runs for the first wicket laid the foundation for the team's commanding victory."